In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
run_label_alignment_fpoh_ohfp_compare.py
----------------------------------------
FP→OH 方向でも「ラベル対応に基づく一致度（ARI）」を定義して算出し、
既存の OH→FP 方向（本文で用いた ARI）と横並び比較する実験スクリプト。

【FP→OH（本スクリプトでの定義）】
- 既存の逆方向サマリ CSV:
    paper_4.2.4.4_OHFP/reverse/analysis_csv/
      Table_4.2.4.4R_fragment-to-OHmajor_{mode}_{index}.csv
  にある各フラグメント行から、
  「FP_cluster（FP 側クラスタID）」と
  「OH_major_material（材料ラベル；fragmentが属するサンプルのOH分布で最頻の材料）」の
  2種類の“ラベル”を取り出す。
- 同じ {mode,index} 内で、全フラグメントについて
  { FP_cluster } と { OH_major_material } の“2つの割当て”の一致度を
  Adjusted Rand Index (ARI) で評価（ラベル対応に相当）。
  → これを ARI_fpoh_fragment_vs_major とする。

【OH→FP（比較対象）】
- 本文の「new vs signless の一致度（ARI）」を利用：
    paper_4.2.4.4_OHFP/main_text/Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv
  から、各 {mode,index,Dataset ∈ {OH,FP}} の ARI を読み、
  {OH,FP} の平均（ARI_mean）を OH→FP の代表値として持ってくる。
  ※ 表に該当行がなければ NaN（スキップ）。

【出力】
ROOT/paper_4.2.4.4_OHFP_20251022/label_alignment_experiment/
  ├─ Table_label_alignment_per_condition.csv
  │    （列：mode,index, ARI_fpoh_fragment_vs_major, ARI_ohfp_mean_from_main, n_fragments）
  ├─ Fig_label_alignment_scatter.png/pdf  （散布図：x=OH→FP(ARI_mean), y=FP→OH(ARI)）
  ├─ Fig_label_alignment_bars.png/pdf     （棒：各条件で2本比較）
  └─ README_label_alignment.json          （参照元パス・件数などのメタ情報）

【実行例】
  # 既定ROOTのまま（ご指定環境）
  python run_label_alignment_fpoh_ohfp_compare.py

  # ルート変更（必要な場合）
  python run_label_alignment_fpoh_ohfp_compare.py \
      --root "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"

  # k≤30 のみ本文表から採用（既定オン）を明示
  python run_label_alignment_fpoh_ohfp_compare.py --use-kle30

  # 全k（appendix表）を使いたい場合（※本文ARIsが無ければ併用して補完）
  python run_label_alignment_fpoh_ohfp_compare.py --use-allk
"""

from __future__ import annotations
import argparse, os, json, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score

# ------------------------------
# 既定のルート（ご指定環境）
# ------------------------------
DEFAULT_ROOT = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")

# パス群
def paths(root: Path):
    base_20251022 = root / "paper_4.2.4.4_OHFP_20251022"
    ohfp_root     = root / "paper_4.2.4.4_OHFP"
    out_root      = base_20251022 / "label_alignment_experiment"
    reverse_csv   = ohfp_root / "reverse" / "analysis_csv"
    main_text     = ohfp_root / "main_text"
    appendix      = ohfp_root / "appendix"
    out_root.mkdir(parents=True, exist_ok=True)
    return {
        "out_root": out_root,
        "reverse_csv": reverse_csv,
        "main_text": main_text,
        "appendix": appendix,
        "base_20251022": base_20251022,
        "ohfp_root": ohfp_root,
    }

# ------------------------------
# ユーティリティ
# ------------------------------
def read_csv_quiet(p: Path) -> pd.DataFrame | None:
    if not p.exists(): return None
    for enc in ("utf-8","utf-8-sig","cp932"):
        try:
            return pd.read_csv(p, encoding=enc)
        except Exception:
            continue
    return None

def find_files_reverse_fragment_to_ohmajor(reverse_csv_dir: Path):
    # Table_4.2.4.4R_fragment-to-OHmajor_{mode}_{index}.csv
    pat = re.compile(
        r"^Table_4\.2\.4\.4R_fragment-to-OHmajor_(?P<mode>top3|cumeig)_(?P<index>silhouette|dunn|gap|ch|db|ptbiserial)\.csv$"
    )
    files=[]
    if reverse_csv_dir.exists():
        for p in reverse_csv_dir.glob("Table_4.2.4.4R_fragment-to-OHmajor_*.csv"):
            m = pat.match(p.name)
            if m:
                files.append((m["mode"], m["index"], p))
    return sorted(files)

def load_maintext_kle30(main_text_dir: Path) -> pd.DataFrame | None:
    # 本文 k≤30 テーブル
    p = main_text_dir / "Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv"
    return read_csv_quiet(p)

def load_appendix_allk(appendix_dir: Path) -> pd.DataFrame | None:
    # 付録 all-k テーブル（必要なら補完に使用）
    p = appendix_dir / "Appendix_Table_allK_OHFP-contrast_summary.csv"
    return read_csv_quiet(p)

def compute_ari_fpoh_fragment_vs_major(df_frag: pd.DataFrame) -> tuple[float,int]:
    """
    df_frag: columns 期待
        - "fragment"（任意）
        - "FP_cluster"（数値 or 文字; 欠損可）
        - "OH_major_material"（文字; 欠損可）
    返り値:
        - ARI（NaN許容）
        - n_frag（計算に使えたフラグメント数）
    """
    if df_frag is None or df_frag.empty:
        return (np.nan, 0)
    df = df_frag.copy()
    # 両方のラベルが有効なものに限定
    df = df.dropna(subset=["FP_cluster","OH_major_material"])
    if df.empty:
        return (np.nan, 0)
    # ラベルを符号化（sklearn は数値・文字いずれもOKだが、安定のため文字化）
    a = df["FP_cluster"].astype(str).values
    b = df["OH_major_material"].astype(str).values
    # 同一 {fragment} に対する 2 partition の ARI
    # ※ fragment ごとの所属（FP_cluster vs OH_major_material）の一致度
    if len(a) < 2 or len(b) < 2:
        return (np.nan, int(len(a)))
    ari = adjusted_rand_score(a, b)
    return (float(ari), int(len(a)))

def extract_ohfp_ari_mean(df_main_like: pd.DataFrame, mode: str, index: str) -> float | None:
    """
    本文（またはall-k）テーブルから {mode,index} の ARI_mean を取得。
      入力テーブルの列仕様想定：
        ["Dataset","mode","index","ARI", ...] （Dataset ∈ {"OH","FP"}）
      → {OH,FP} の ARI を同じ {mode,index} で平均。
    """
    if df_main_like is None or df_main_like.empty:
        return None
    sl = df_main_like[(df_main_like["mode"]==mode) & (df_main_like["index"]==index)]
    if sl.empty: return None
    # Dataset ごとに代表（同じ条件に複数行があり得る場合は平均）
    tbl = (sl.groupby("Dataset")["ARI"].mean().reindex(["OH","FP"]))
    if tbl.isna().all():
        return None
    return float(tbl.mean(skipna=True))

def bar_two(ax, x_idx, v1, v2, w=0.36, labels=("OH→FP (ARI_mean)","FP→OH (ARI)")):
    x = np.arange(len(x_idx))
    ax.bar(x - w/2, v1, width=w, alpha=0.9, label=labels[0])
    ax.bar(x + w/2, v2, width=w, alpha=0.9, label=labels[1])
    ax.set_xticks(x); ax.set_xticklabels(x_idx, rotation=35, ha="right")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)

# ------------------------------
# メイン
# ------------------------------
def main():
    ap = argparse.ArgumentParser(description="FP→OHラベル対応（ARI）とOH→FP（本文ARI）の比較")
    ap.add_argument("--root", type=str, default=str(DEFAULT_ROOT), help="解析ルート（paper_4.2.4.4_OHFP_* がある場所）")
    ap.add_argument("--use-kle30", action="store_true", help="本文（k≤30）のARIsのみを使う（既定）")
    ap.add_argument("--use-allk", action="store_true", help="付録（all-k）を使って補完する")
    args, _ = ap.parse_known_args()  # Jupyter耐性

    root = Path(args.root).resolve()
    P = paths(root)
    out_root = P["out_root"]

    # 逆方向 fragment→OHmajor のファイル列挙
    rev_list = find_files_reverse_fragment_to_ohmajor(P["reverse_csv"])
    if not rev_list:
        raise FileNotFoundError(f"逆方向CSVが見つかりません: {P['reverse_csv']}")

    # 本文・付録テーブル（OH→FP）
    df_main = load_maintext_kle30(P["main_text"])
    df_allk = load_appendix_allk(P["appendix"])
    use_allk = bool(args.use_allk)
    use_kle30 = bool(args.use_kle30) or (not use_allk)  # 既定は k≤30 優先

    rows=[]
    for mode, index, fpath in rev_list:
        df_frag = read_csv_quiet(fpath)
        ari_fpoh, n_frag = compute_ari_fpoh_fragment_vs_major(df_frag)

        # OH→FP 側の ARI_mean を取得（本文優先、必要なら all-k で補完）
        ari_ohfp = None
        if use_kle30 and df_main is not None:
            ari_ohfp = extract_ohfp_ari_mean(df_main, mode, index)
        if ari_ohfp is None and use_allk and df_allk is not None:
            ari_ohfp = extract_ohfp_ari_mean(df_allk, mode, index)

        rows.append({
            "mode": mode,
            "index": index,
            "n_fragments": n_frag,
            "ARI_fpoh_fragment_vs_major": ari_fpoh,
            "ARI_ohfp_mean_from_main": ari_ohfp
        })

    T = pd.DataFrame(rows).sort_values(["mode","index"]).reset_index(drop=True)

    # 保存
    tab_out = out_root / "Table_label_alignment_per_condition.csv"
    T.to_csv(tab_out, index=False, encoding="utf-8-sig")

    # 図1：散布図（x=OH→FP(ARI_mean), y=FP→OH(ARI)）
    fig1, ax1 = plt.subplots(figsize=(7.5,6), dpi=300)
    x = pd.to_numeric(T["ARI_ohfp_mean_from_main"], errors="coerce")
    y = pd.to_numeric(T["ARI_fpoh_fragment_vs_major"], errors="coerce")
    m = x.notna() & y.notna()
    ax1.scatter(x[m], y[m], alpha=0.9)
    # 近似直線（点が2つ以上ある場合）
    if m.sum() >= 2:
        coef = np.polyfit(x[m].values, y[m].values, 1)
        xs = np.linspace(float(np.nanmin(x[m])), float(np.nanmax(x[m])), 100)
        ys = np.polyval(coef, xs)
        ax1.plot(xs, ys)
        ax1.text(0.02, 0.96, f"y = {coef[0]:.2f} x + {coef[1]:.2f}", transform=ax1.transAxes,
                 ha="left", va="top", fontsize=9)
    ax1.set_xlabel("OH→FP alignment (ARI mean from main)")
    ax1.set_ylabel("FP→OH alignment (ARI between FP_cluster and OH_major)")
    ax1.set_title("Label alignment: OH→FP vs FP→OH (per mode/index)")
    ax1.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
    fig1.tight_layout()
    fig1.savefig(out_root / "Fig_label_alignment_scatter.png", bbox_inches="tight")
    fig1.savefig(out_root / "Fig_label_alignment_scatter.pdf", bbox_inches="tight")
    plt.close(fig1)

    # 図2：棒（各条件で2本）
    fig2, ax2 = plt.subplots(figsize=(11,5.5), dpi=300)
    idx_lbl = [f"{m}-{i}" for m,i in zip(T["mode"], T["index"])]
    v1 = pd.to_numeric(T["ARI_ohfp_mean_from_main"], errors="coerce").fillna(np.nan)
    v2 = pd.to_numeric(T["ARI_fpoh_fragment_vs_major"], errors="coerce").fillna(np.nan)
    bar_two(ax2, idx_lbl, v1, v2)
    ax2.set_title("Label alignment per condition")
    ax2.legend(loc="upper left", bbox_to_anchor=(1.02,1.0))
    fig2.tight_layout()
    fig2.savefig(out_root / "Fig_label_alignment_bars.png", bbox_inches="tight")
    fig2.savefig(out_root / "Fig_label_alignment_bars.pdf", bbox_inches="tight")
    plt.close(fig2)

    # README
    meta = {
        "root": str(root),
        "reverse_fragment_dir": str(P["reverse_csv"]),
        "main_text_dir": str(P["main_text"]),
        "appendix_dir": str(P["appendix"]),
        "used_kle30": bool(use_kle30),
        "used_allk": bool(use_allk),
        "n_conditions_found": int(len(T)),
        "n_with_both_scores": int(int((T["ARI_ohfp_mean_from_main"].notna() & T["ARI_fpoh_fragment_vs_major"].notna()).sum())),
        "outputs": {
            "table": str(tab_out),
            "fig_scatter": str(out_root / "Fig_label_alignment_scatter.png"),
            "fig_bars": str(out_root / "Fig_label_alignment_bars.png"),
        }
    }
    (out_root / "README_label_alignment.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[SAVE]", tab_out)
    print("[SAVE]", out_root / "Fig_label_alignment_scatter.png")
    print("[SAVE]", out_root / "Fig_label_alignment_bars.png")
    print("[SAVE]", out_root / "README_label_alignment.json")
    print("\n✅ Done. outputs ->", out_root)

if __name__ == "__main__":
    main()


[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/label_alignment_experiment/legacy/Table_label_alignment_per_condition.csv
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/label_alignment_experiment/legacy/Fig_label_alignment_scatter.png
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/label_alignment_experiment/legacy/Fig_label_alignment_bars.png
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/label_alignment_experiment/legacy/README_label_alignment.json

✅ Done. outputs -> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/label_alignment_experiment/legacy


In [12]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
run_label_alignment_fpoh_ohfp_compare_kmodes.py
-----------------------------------------------
FP→OH 方向でも「ラベル対応（fragment の FP_cluster vs OH_major_material）」に基づく
一致度（Adjusted Rand Index; ARI）を算出し、既存の OH→FP 方向（本文の ARI）と
横並び比較します。さらに下記 3 モードで条件集合を切り替え、結果を別フォルダに出力します。

  k-mode:
    1) legacy   : 「従来の絞り込み」… 本文テーブル（k≤30, main_text）の条件のみで比較
    2) allk     : 「足切りを外す」… 付録テーブル（all-k, appendix）の条件で比較
    3) unify30  : 「reverse 側も k≤30 に統一」… 付録 all-k から OH と FP がともに k≤30
                  を満たす {mode,index} のみを比較（条件集合の統一）

入力（既存の出力を再利用）:
  ROOT/paper_4.2.4.4_OHFP/
    ├─ main_text/Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv
    ├─ appendix/Appendix_Table_allK_OHFP-contrast_summary.csv
    └─ reverse/analysis_csv/
         Table_4.2.4.4R_fragment-to-OHmajor_{mode}_{index}.csv

出力:
  ROOT/paper_4.2.4.4_OHFP_20251022/label_alignment_experiment/<k-mode>/
    ├─ Table_label_alignment_per_condition.csv
    │     (mode,index,n_fragments,ARI_fpoh_fragment_vs_major,ARI_ohfp_mean_from_main)
    ├─ Fig_label_alignment_scatter.(png|pdf)
    ├─ Fig_label_alignment_bars.(png|pdf)
    └─ README_label_alignment.json

使い方:
  # 3モードを一括実行
  python run_label_alignment_fpoh_ohfp_compare_kmodes.py --run-all

  # 個別モード
  python run_label_alignment_fpoh_ohfp_compare_kmodes.py --k-mode legacy
  python run_label_alignment_fpoh_ohfp_compare_kmodes.py --k-mode allk
  python run_label_alignment_fpoh_ohfp_compare_kmodes.py --k-mode unify30

  # ルート変更が必要な場合
  python run_label_alignment_fpoh_ohfp_compare_kmodes.py --root "/path/to/cal_cluster_No" --k-mode legacy
"""

from __future__ import annotations
import argparse, json, re
from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.metrics import adjusted_rand_score

# -------------------------------------------------
# 既定のルート（ご指定環境）
# -------------------------------------------------
DEFAULT_ROOT = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")

# -------------------------------------------------
# パス管理
# -------------------------------------------------
def build_paths(root: Path, k_mode: str):
    base_20251022 = root / "paper_4.2.4.4_OHFP_20251022"
    out_root = base_20251022 / "label_alignment_experiment" / k_mode
    ohfp_root = root / "paper_4.2.4.4_OHFP"
    reverse_csv = ohfp_root / "reverse" / "analysis_csv"
    main_text = ohfp_root / "main_text"
    appendix  = ohfp_root / "appendix"
    out_root.mkdir(parents=True, exist_ok=True)
    return {
        "out_root": out_root,
        "reverse_csv": reverse_csv,
        "main_text": main_text,
        "appendix": appendix,
        "ohfp_root": ohfp_root,
        "base_20251022": base_20251022,
    }

# -------------------------------------------------
# IO ユーティリティ
# -------------------------------------------------
def read_csv_quiet(p: Path) -> pd.DataFrame | None:
    if not p.exists(): return None
    for enc in ("utf-8", "utf-8-sig", "cp932"):
        try:
            return pd.read_csv(p, encoding=enc)
        except Exception:
            continue
    return None

def find_files_reverse_fragment_to_ohmajor(reverse_csv_dir: Path):
    pat = re.compile(
        r"^Table_4\.2\.4\.4R_fragment-to-OHmajor_"
        r"(?P<mode>top3|cumeig)_(?P<index>silhouette|dunn|gap|ch|db|ptbiserial)\.csv$"
    )
    files=[]
    if reverse_csv_dir.exists():
        for p in reverse_csv_dir.glob("Table_4.2.4.4R_fragment-to-OHmajor_*.csv"):
            m = pat.match(p.name)
            if m:
                files.append((m["mode"], m["index"], p))
    return sorted(files)

def load_maintext_kle30(main_text_dir: Path) -> pd.DataFrame | None:
    p = main_text_dir / "Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv"
    return read_csv_quiet(p)

def load_appendix_allk(appendix_dir: Path) -> pd.DataFrame | None:
    p = appendix_dir / "Appendix_Table_allK_OHFP-contrast_summary.csv"
    return read_csv_quiet(p)

# -------------------------------------------------
# 計算：FP→OH 側の ARI（fragment の FP_cluster vs OH_major_material）
# -------------------------------------------------
def compute_ari_fpoh_fragment_vs_major(df_frag: pd.DataFrame) -> tuple[float,int]:
    """
    df_frag 期待列: FP_cluster, OH_major_material
    返り値: (ARI, n_frag_used)
    """
    if df_frag is None or df_frag.empty:
        return (np.nan, 0)
    df = df_frag.dropna(subset=["FP_cluster", "OH_major_material"]).copy()
    if df.empty: return (np.nan, 0)
    a = df["FP_cluster"].astype(str).values
    b = df["OH_major_material"].astype(str).values
    if len(a) < 2 or len(b) < 2:
        return (np.nan, int(len(a)))
    return (float(adjusted_rand_score(a, b)), int(len(a)))

# -------------------------------------------------
# 計算：OH→FP 側の ARI_mean（本文/付録テーブルから抽出）
# -------------------------------------------------
def extract_ari_mean_from_table(df_main_like: pd.DataFrame, mode: str, index: str) -> float | None:
    """
    期待列: ["Dataset","mode","index","ARI", ...]  Dataset ∈ {"OH","FP"}
    同一 {mode,index} の OH と FP の ARI を平均して ARI_mean として返す。
    """
    if df_main_like is None or df_main_like.empty:
        return None
    sl = df_main_like[(df_main_like["mode"]==mode) & (df_main_like["index"]==index)]
    if sl.empty: return None
    tbl = sl.groupby("Dataset")["ARI"].mean().reindex(["OH","FP"])
    if tbl.isna().all(): return None
    return float(tbl.mean(skipna=True))

# -------------------------------------------------
# k-mode ごとの「比較対象となる {mode,index} 条件集合」の作成
# -------------------------------------------------
def condition_keys_for_k_mode(k_mode: str,
                              df_main: pd.DataFrame | None,
                              df_allk: pd.DataFrame | None) -> set[tuple[str,str]]:
    """
    - legacy : main_text（k≤30）に現れる {mode,index}
    - allk   : appendix（all-k）に現れる {mode,index}
    - unify30: appendix（all-k）のうち、OHとFPがともに k≤30 の {mode,index}
              （OH側のレコードとFP側のレコードが存在し、各々の kA/kB が30以下）
    """
    def pairs_from(df: pd.DataFrame | None) -> set[tuple[str,str]]:
        if df is None or df.empty: return set()
        need_cols = {"mode","index"}
        if not need_cols.issubset(df.columns):
            return set()
        X = df[["mode","index"]].dropna().drop_duplicates()
        return set((str(r.mode), str(r.index)) for r in X.itertuples())

    if k_mode == "legacy":
        return pairs_from(df_main)

    if k_mode == "allk":
        return pairs_from(df_allk)

    if k_mode == "unify30":
        if df_allk is None or df_allk.empty:
            return set()
        need = {"Dataset","mode","index","kA","kB"}
        if not need.issubset(df_allk.columns):
            return set()
        oh_ok = df_allk[(df_allk["Dataset"]=="OH") & (pd.to_numeric(df_allk["kA"], errors="coerce") <= 30)][["mode","index"]]
        fp_ok = df_allk[(df_allk["Dataset"]=="FP") & (pd.to_numeric(df_allk["kB"], errors="coerce") <= 30)][["mode","index"]]
        if oh_ok.empty or fp_ok.empty:
            return set()
        oh_set = set((str(r.mode), str(r.index)) for r in oh_ok.dropna().drop_duplicates().itertuples())
        fp_set = set((str(r.mode), str(r.index)) for r in fp_ok.dropna().drop_duplicates().itertuples())
        return oh_set.intersection(fp_set)

    return set()

# -------------------------------------------------
# 可視化
# -------------------------------------------------
def plot_scatter(T: pd.DataFrame, out_png: Path, out_pdf: Path, title_suffix: str):
    fig, ax = plt.subplots(figsize=(7.5,6), dpi=300)
    x = pd.to_numeric(T["ARI_ohfp_mean_from_main"], errors="coerce")
    y = pd.to_numeric(T["ARI_fpoh_fragment_vs_major"], errors="coerce")
    m = x.notna() & y.notna()
    ax.scatter(x[m], y[m], alpha=0.9)
    if m.sum() >= 2:
        coef = np.polyfit(x[m].values, y[m].values, 1)
        xs = np.linspace(float(np.nanmin(x[m])), float(np.nanmax(x[m])), 100)
        ys = np.polyval(coef, xs)
        ax.plot(xs, ys)
        ax.text(0.02, 0.96, f"y = {coef[0]:.2f} x + {coef[1]:.2f}", transform=ax.transAxes,
                ha="left", va="top", fontsize=9)
    ax.set_xlabel("OH→FP alignment (ARI mean from main/all-k)")
    ax.set_ylabel("FP→OH alignment (ARI between FP_cluster and OH_major)")
    ax.set_title(f"Label alignment: OH→FP vs FP→OH ({title_suffix})")
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)

def plot_bars(T: pd.DataFrame, out_png: Path, out_pdf: Path, title_suffix: str):
    fig, ax = plt.subplots(figsize=(11,5.5), dpi=300)
    idx_lbl = [f"{m}-{i}" for m,i in zip(T["mode"], T["index"])]
    v1 = pd.to_numeric(T["ARI_ohfp_mean_from_main"], errors="coerce")
    v2 = pd.to_numeric(T["ARI_fpoh_fragment_vs_major"], errors="coerce")
    x = np.arange(len(idx_lbl)); w=0.36
    ax.bar(x - w/2, v1, width=w, alpha=0.9, label="OH→FP (ARI_mean)")
    ax.bar(x + w/2, v2, width=w, alpha=0.9, label="FP→OH (ARI)")
    ax.set_xticks(x); ax.set_xticklabels(idx_lbl, rotation=35, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Label alignment per condition ({title_suffix})")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02,1.0))
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)

# -------------------------------------------------
# 1モード分の処理本体
# -------------------------------------------------
def run_one_mode(root: Path, k_mode: str):
    P = build_paths(root, k_mode)
    out_root = P["out_root"]

    # 入力テーブル（OH→FP）
    df_main = load_maintext_kle30(P["main_text"])     # k≤30
    df_allk = load_appendix_allk(P["appendix"])       # all-k

    # 比較対象の条件集合
    cond_keys = condition_keys_for_k_mode(k_mode, df_main, df_allk)
    if not cond_keys:
        raise RuntimeError(f"{k_mode}: 比較対象の (mode,index) が見つかりません。入力表をご確認ください。")

    # 逆方向 fragment→OHmajor
    rev_files = { (m,i): f for (m,i,f) in find_files_reverse_fragment_to_ohmajor(P["reverse_csv"]) }

    rows=[]
    for (mode, index) in sorted(cond_keys):
        df_frag = read_csv_quiet(rev_files.get((mode,index), Path("dummy")))
        ari_fpoh, n_frag = compute_ari_fpoh_fragment_vs_major(df_frag) if df_frag is not None else (np.nan, 0)

        if k_mode == "legacy":
            ari_ohfp = extract_ari_mean_from_table(df_main, mode, index)
        elif k_mode in ("allk", "unify30"):
            ari_ohfp = extract_ari_mean_from_table(df_allk, mode, index)
        else:
            ari_ohfp = None

        rows.append({
            "mode": mode,
            "index": index,
            "n_fragments": n_frag,
            "ARI_fpoh_fragment_vs_major": ari_fpoh,
            "ARI_ohfp_mean_from_main": ari_ohfp
        })

    T = pd.DataFrame(rows).sort_values(["mode","index"]).reset_index(drop=True)

    # 保存
    tab_out = out_root / "Table_label_alignment_per_condition.csv"
    T.to_csv(tab_out, index=False, encoding="utf-8-sig")

    # 図
    plot_scatter(
        T,
        out_png = out_root / "Fig_label_alignment_scatter.png",
        out_pdf = out_root / "Fig_label_alignment_scatter.pdf",
        title_suffix = f"k-mode = {k_mode}"
    )
    plot_bars(
        T,
        out_png = out_root / "Fig_label_alignment_bars.png",
        out_pdf = out_root / "Fig_label_alignment_bars.pdf",
        title_suffix = f"k-mode = {k_mode}"
    )

    # README
    meta = {
        "root": str(root),
        "k_mode": k_mode,
        "reverse_fragment_dir": str(P["reverse_csv"]),
        "main_text_dir": str(P["main_text"]),
        "appendix_dir": str(P["appendix"]),
        "n_conditions": int(len(T)),
        "n_with_both_scores": int((T["ARI_ohfp_mean_from_main"].notna() & T["ARI_fpoh_fragment_vs_major"].notna()).sum()),
        "outputs": {
            "table": str(tab_out),
            "fig_scatter": str(out_root / "Fig_label_alignment_scatter.png"),
            "fig_bars": str(out_root / "Fig_label_alignment_bars.png"),
        }
    }
    (out_root / "README_label_alignment.json").write_text(
        json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    print("[SAVE]", tab_out)
    print("[SAVE]", out_root / "Fig_label_alignment_scatter.png")
    print("[SAVE]", out_root / "Fig_label_alignment_bars.png")
    print("[SAVE]", out_root / "README_label_alignment.json")
    print(f"\n✅ Done. outputs -> {out_root}")

# -------------------------------------------------
# MAIN
# -------------------------------------------------
def main():
    ap = argparse.ArgumentParser(description="FP→OH（ラベル対応ARI）とOH→FP（本文ARI）の比較（3つのk-mode切替）")
    ap.add_argument("--root", type=str, default=str(DEFAULT_ROOT), help="解析ルート（paper_4.2.4.4_OHFP_* がある場所）")
    ap.add_argument("--k-mode", type=str, default="legacy", choices=["legacy","allk","unify30"],
                    help="legacy:本文(k≤30)/ allk:付録(all-k)/ unify30:all-kからOHとFPのk≤30の条件に統一")
    ap.add_argument("--run-all", action="store_true", help="legacy / allk / unify30 を連続実行")
    args, _ = ap.parse_known_args()  # Jupyter耐性

    root = Path(args.root).resolve()

    if args.run_all:
        for m in ["legacy", "allk", "unify30"]:
            print(f"\n=== k-mode: {m} ===")
            run_one_mode(root, m)
    else:
        run_one_mode(root, args.k_mode)

if __name__ == "__main__":
    main()


[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/label_alignment_experiment/legacy/Table_label_alignment_per_condition.csv
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/label_alignment_experiment/legacy/Fig_label_alignment_scatter.png
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/label_alignment_experiment/legacy/Fig_label_alignment_bars.png
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/label_alignment_experiment/legacy/README_label_alignment.json

✅ Done. outputs -> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/label_alignment_experiment/legacy


In [13]:
# 1) appendix の all-k テーブルの有無
ls -l /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/appendix/Appendix_Table_allK_OHFP-contrast_summary.csv

# 2) reverse/analysis_csv の fragment-to-OHmajor_* 一覧
ls -l /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/reverse/analysis_csv/Table_4.2.4.4R_fragment-to-OHmajor_*.csv


SyntaxError: invalid decimal literal (3183791846.py, line 2)

In [14]:
# 3モードまとめて
python run_label_alignment_fpoh_ohfp_compare_kmodes.py --run-all

# または個別に
python run_label_alignment_fpoh_ohfp_compare_kmodes.py --k-mode allk
python run_label_alignment_fpoh_ohfp_compare_kmodes.py --k-mode unify30


SyntaxError: invalid syntax (456815270.py, line 2)

In [15]:
def run_one_mode(root: Path, k_mode: str):
    P = build_paths(root, k_mode)
    out_root = P["out_root"]

    df_main = load_maintext_kle30(P["main_text"])
    df_allk = load_appendix_allk(P["appendix"])

    cond_keys = condition_keys_for_k_mode(k_mode, df_main, df_allk)
-   if not cond_keys:
-       raise RuntimeError(f"{k_mode}: 比較対象の (mode,index) が見つかりません。入力表をご確認ください。")
+   if not cond_keys:
+       # 説明だけ書いて終了（フォルダは作る）
+       meta = {
+           "root": str(root),
+           "k_mode": k_mode,
+           "reason": "no (mode,index) candidates",
+           "hint": {
+               "need_files": {
+                   "legacy": str(P["main_text"] / "Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv"),
+                   "allk":   str(P["appendix"] / "Appendix_Table_allK_OHFP-contrast_summary.csv"),
+                   "unify30": "all-k表に Dataset,mode,index,kA,kB 列が必要（OHとFPともに k<=30）"
+               }
+           }
+       }
+       (out_root / "README_label_alignment.json").write_text(
+           json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8"
+       )
+       print(f"[WARN] {k_mode}: 比較対象の (mode,index) が見つかりません。README を出力してスキップします -> {out_root}")
+       return


SyntaxError: invalid syntax (3981344050.py, line 9)

In [16]:
python run_label_alignment_fpoh_ohfp_compare_kmodes.py --run-all


SyntaxError: invalid syntax (2370937939.py, line 1)

In [17]:
双方向フレーム（OH→FP / FP→OH）と指標の役割分担

データ・出力物（joined表・図表群）の位置づけ

指標定義（FP→OH：6指標）
の再構成もお願い

合成スコア（composite）の作り方
の再構成もお願い

閾値を使う箇所と根拠（cohesive判定）
の再構成もお願い

解析条件と選定理由（cumeig、NbClustのgap/dbなど）
の再構成もお願い

結果①：OH→FP（ARI）
の再構成もお願い

結果②：FP→OH（purity/entropyの材料散布）
の再構成もお願い

結果③：OHクラスタのcohesive ratio（dbで最大化）
の再構成もお願い

補足結果：材料ペア指標（FP_major_same、cosine、JS）

由来別比較（inM/ip1M/ip2M/nM/p1M/p2M）

解釈・非対称性の注意（ARIとcompositeの補完関係）

まとめ（要点と次節への接続）

SyntaxError: invalid character '（' (U+FF08) (4161802587.py, line 1)